# Script 10 - Mplus results

Parses every `.out` file in `scripts/9_mplus_models` with `scripts/mplus_parse.py`
and builds the Mplus-derived tables and figures through `scripts/mplus_report.py`.
Running this notebook and running `python mplus_report.py` write the same files;
the notebook also prints Table 2 and runs the repository checkers.

| output | contents |
|---|---|
| `result/raw/mplus_paths_long.csv`, `mplus_effects_long.csv`, `mplus_estimates_all.csv` | parsed long-format frames |
| `result/qc/QC_Mplus_Internal.xlsx` | internal: every parameter, every check, provenance |
| `result/main/Table2_SEM_Mediation.{csv,xlsx}` | manuscript Table 2, with its Scale and Notes sheets |
| `result/main/Table3_Age_Moderation.{csv,xlsx}` | manuscript Table 3, with its Scale and Notes sheets |
| `result/supplementary/Supp_Tables.xlsx` | Supplementary Tables S3-S7, every model |
| `figure/main/Fig2_MediatorContrast.{pdf,png}` | a-paths and indirect associations, KDM-BA against log SII |
| `figure/main/Fig4_Age_Moderation_SimpleSlopes.{pdf,png}` | simple slopes of the rhythm-to-biological-age association by age |
| `figure/main/Fig3_PathDecomposition.{pdf,png}` | full path decomposition, all three metrics |
| `figure/supplementary/SuppFigS5_RhythmToBiologicalAge_VolumeAdjustment.{pdf,png}` | a1 across metrics and cells, core against volume-adjusted covariates |
| `figure/source_data/Source_Data_*.xlsx` | one per figure |
| `DISPLAY_ITEMS.md`, `VARIABLES.md`, `CAPTIONS.md`, Supplementary Table S16 | regenerated by the checkers in Step 5 |

Rules applied in the run: every cognition model must have positive factor
loadings before any COG coefficient is read; Holm correction is applied across
the two mediators within each model; duplicate specifications are detected from
n / a1 / b1 rather than from filenames; the mediation proportion is suppressed
where the total or indirect effect is not significant or the two have opposite
signs; any REVIEW row in the QC checks stops the notebook.

## Step 0 - Imports and paths

Locate the project root, import the project modules and create the output directories.

In [ ]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

def find_root():
    """Locate the project root: NHANES_SEM_ROOT, then the working directory and its parents."""
    env = os.environ.get('NHANES_SEM_ROOT')
    cands = ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]
    for p in cands:
        if (p / 'data' / 'processed').is_dir() and (p / 'scripts').is_dir():
            return p
    raise RuntimeError(
        'cannot find the project root from ' + str(Path.cwd())
        + ' -- start Jupyter inside the project, or set NHANES_SEM_ROOT')

BASE = find_root()
sys.path.insert(0, str(BASE / 'scripts'))

# Reload the project modules so a running kernel sees the current code:
# nature_palette first, because the others import from it, then parse,
# export and report in dependency order.
import importlib
import nature_palette
importlib.reload(nature_palette)
import mplus_parse, mplus_export, mplus_report
for _m in (nature_palette, mplus_parse, mplus_export, mplus_report):
    importlib.reload(_m)

from mplus_parse import parse_dir, label_models
from mplus_export import (flag_duplicates, path_table, effect_table,
                          interaction_table, indirect_with_holm,
                          mediation_proportion, fmt_ci, fmt_p)

R_MAIN = BASE / 'result' / 'main'
R_RAW  = BASE / 'result' / 'raw'
FIG    = BASE / 'figure' / 'main'
SRC    = BASE / 'figure' / 'source_data'
for p in (R_MAIN, R_RAW, FIG, SRC):
    p.mkdir(parents=True, exist_ok=True)


## Step 1 - Parse every .out file

Parse the `.out` files into long-format frames, label and de-duplicate the
models, check factor orientation, and write the three long-format CSVs to
`result/raw/`.

In [ ]:
# Parse every .out file into long-format frames.
est, eff, info = parse_dir(BASE / 'scripts' / '9_mplus_models')
info    = flag_duplicates(est, label_models(info))
paths   = path_table(est, info)
effects = effect_table(eff, info)
inter   = interaction_table(est, info)
holm    = indirect_with_holm(effects)
props   = mediation_proportion(est, effects)

print(f'models {len(info)}   parameters {len(est):,}   effects {len(eff):,}')

# Factor orientation must be positive before any COG coefficient is read.
bad = info[info.loadings_positive == False]
assert not len(bad), (
    f'negative factor loadings in {list(bad.model)} - every coefficient on COG '
    f'is sign-flipped there')
print('factor orientation: all cognition models positive')

if info.is_duplicate.any():
    print('duplicate specifications (counted once downstream):')
    for _, r in info[info.is_duplicate].iterrows():
        print(f'  {r.model} == {r.duplicate_of}')

paths.to_csv(R_RAW / 'mplus_paths_long.csv', index=False)
effects.to_csv(R_RAW / 'mplus_effects_long.csv', index=False)
est.to_csv(R_RAW / 'mplus_estimates_all.csv', index=False)
print(f'wrote three long-format files to {R_RAW}')


## Step 2 - Table 2, mediation across every rhythm metric

Build Table 2 with `mplus_report.table_main_mediation()` and print it: one block
per analysis cell and, within each block, one sub-block per rhythm metric
(intradaily variability, interdaily stability, relative amplitude). The table is
written to disk in Step 4 by `write_main()`.

In [ ]:
# Table 2 is built by mplus_report.table_main_mediation, the same function
# write_main() uses in Step 4; the Notes sheet is written there via build_all().
from mplus_report import (table_main_mediation, cell_scales,
                          EXPOSURES)  # reloaded in Step 0

t4 = table_main_mediation(paths, effects, holm, props)

# Every modelled exposure must appear in the table.
shown = {e for e in EXPOSURES
         if t4['Rhythm metric'].str.contains(f'({e})', regex=False).any()}
assert shown == set(EXPOSURES), f'missing exposures: {set(EXPOSURES) - shown}'
assert not t4.empty, 'Table 2 came back empty'

print(t4.to_string(index=False))
print(f'\n{len(t4)} rows: {len(EXPOSURES)} rhythm metrics x 3 analysis cells')
print('This table is written to disk as Table 2 by write_main() in Step 4, '
      'together with its Scale and Notes sheets. It is not written here: one '
      'table, one producer.')


## Steps 3 and 4 - Figures and workbooks

Both are produced by one call to `mplus_report.build_all()`, which writes the
four figures, the QC workbook, Table 2, Table 3 and the supplementary tables.
Manuscript numbering is defined in `mplus_report.FIGNUM` and `mplus_report.TABNUM`.

| number | figure | produced by |
|---|---|---|
| Fig 1 | study overview and sample flow | script 14 |
| Fig 2 | mediator contrast: a-paths and indirect associations, KDM-BA against log SII | `mplus_report.fig_dissociation` |
| Fig 3 | age moderation, simple slopes | `mplus_report.fig2_moderation` |
| Fig 4 | full path decomposition | `mplus_report.fig3_paths` |
| Supp Fig S5 | rhythm to biological age, core against volume-adjusted covariates | `mplus_report.fig1_specificity` |

Each figure is written as PDF (vector, editable text) and 600 dpi PNG at 183 mm
double-column width, with its own source-data workbook.

The QC sheet is the gate: the run raises if any check reads REVIEW, or if the
figures and tables produced do not match `FIGNUM` and `TABNUM`.

## Run

In [ ]:
from mplus_report import build_all, FIGNUM, TABNUM

out = build_all()

# QC gate: any REVIEW row stops the run.
review = out['checks'][out['checks'].status == 'REVIEW']
assert not len(review), (
    'QC checks flagged for review:\n'
    + '\n'.join(f'  {r.check}: {r.detail}' for _, r in review.iterrows()))

# The figures and tables produced must match FIGNUM and TABNUM.
assert set(out['figures']) == set(FIGNUM.values()), (
    f"figures produced {sorted(out['figures'])} do not match FIGNUM "
    f'{sorted(FIGNUM.values())}')
produced_tables = {p.stem for p in out['main_tables']}
assert produced_tables == set(TABNUM.values()), (
    f'main tables {sorted(produced_tables)} do not match TABNUM '
    f'{sorted(TABNUM.values())}')

print(f"\n{out['n_models']} models parsed; "
      f"{len(out['checks'])}/{len(out['checks'])} QC checks pass")


## Step 5 - Run the checkers and regenerate the derived documents

Runs `verify_figures.py`, `display_map.py --write`, `variable_dictionary.py --write`,
`captions.py --write` and `verify_citations.py` as subprocesses. They regenerate
`DISPLAY_ITEMS.md`, `VARIABLES.md`, `CAPTIONS.md` and Supplementary Table S16;
the cell raises if any checker exits non-zero.

In [ ]:
# Run the checkers and regenerate DISPLAY_ITEMS.md / CAPTIONS.md / VARIABLES.md.
# Each checker is run as a subprocess: each has its own main() that calls
# sys.exit(), and a subprocess reads the files just written to disk.
import subprocess, sys
from pathlib import Path

_SCRIPTS = Path(find_root()) / 'scripts'

_STEPS = [
    ('verify_figures.py',      [],           'Nature figure spec + source data'),
    ('display_map.py',         ['--write'],  'display items -> producing script'),
    ('variable_dictionary.py', ['--write'],  'Supp Table S16 + VARIABLES.md'),
    ('captions.py',            ['--write'],  'titles + captions -> CAPTIONS.md'),
    ('verify_citations.py',    [],           'every display item cited in the body'),
]

_failed = []
for _script, _args, _what in _STEPS:
    _r = subprocess.run([sys.executable, _script, *_args],
                        cwd=_SCRIPTS, capture_output=True, text=True)
    _tail = (_r.stdout or _r.stderr).strip().splitlines()
    print(f'{_script:24s} {"ok  " if _r.returncode == 0 else "FAIL"}  {_what}')
    print(f'{"":26s}{_tail[-1][:88] if _tail else ""}')
    if _r.returncode != 0:
        _failed.append((_script, (_r.stdout + _r.stderr)[-1500:]))

if _failed:
    raise RuntimeError(
        'checkers failed:\n\n' +
        '\n\n'.join(f'=== {s} ===\n{o}' for s, o in _failed))

print('\nall checkers pass; DISPLAY_ITEMS.md, VARIABLES.md and CAPTIONS.md '
      'are current')
